# 02 Preprocessing: from the clean dataset to training and test data

`cleaned_all.csv` --> keep the allowed columns --> time control as dummies --> split by game into training and test games --> save `train.csv` and `test.csv` --> explore the training data --> check the feature groups --> define the preprocessing that is learned inside the cross-validation

Every row is one position (one move) of a Lichess game. This notebook only prepares the data. The models are in `03_models.ipynb`.

**Two kinds of preprocessing** (deck 06, notebook 209)
- Steps that treat every row on its own and learn nothing from other rows: keeping the allowed columns, dummies for the three fixed time controls. We do them here, once, for all rows.
- Steps that learn numbers from the data: the median that fills missing values, the mean and the standard deviation for standardization. These numbers must come from the training data only ("estimate, then freeze", deck 06), and inside the cross-validation from the training part of each fold. We explain and define them here (section 8), but they are fitted in `03_models.ipynb`.

**Rules**
- We split off the test games right after the dummies (section 4) and do not look at them again. Everything after that uses the training games only (notebook 209: "split off the test set before you explore").
- All positions of one game stay on the same side of the split (deck P1: split by group).
- `eval_loss_cp`, `played_eval_cp` and `played_mate` are never features: they are computed from the played move and give away the target (deck P1: "the answer in disguise").

In [ ]:
#import
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

#use a nicer style for plots (as in the course notebooks)
plt.style.use("seaborn-v0_8-muted")

## 1. Data import

We read the clean dataset that `01_cleaning.ipynb` saved.

In [ ]:
#game_id is read as text, so that an id like "0001" is never turned into the number 1
master = pd.read_csv("../data/processed/cleaned_all.csv", dtype={"game_id": str})

print(f"{master['game_id'].nunique()} games, {len(master)} positions, {master.shape[1]} columns")

## 2. Keep only the allowed columns

We write the 60 predictors out by name, in the order of `docs/FEATURE_DICTIONARY.md`. We never build the features as "all columns except y", because some metadata give away the target (README).

We keep `y`, the 60 predictors, `pre_eval_cp_missing` (made in `01_cleaning`) and three metadata columns:
- `game_id`: to split by game,
- `source_pgn`: the raw file, one per time control, to balance the split,
- `time_control`: for the dummies in section 3.

This step learns nothing from the data, so it may happen before the split.

In [ ]:
#the 60 candidate predictors (docs/FEATURE_DICTIONARY.md)
predictors = [
    #move features: the legal moves before the played move
    "legal_move_count", "capture_move_count", "checking_move_count", "checking_capture_count",
    "quiet_move_count", "promotion_move_count", "castling_move_count",
    #board features
    "in_check", "checker_count",
    "material_balance_cp", "total_material_cp", "own_non_pawn_material_cp", "opponent_non_pawn_material_cp",
    "own_pawn_count", "opponent_pawn_count", "own_knight_count", "opponent_knight_count",
    "own_bishop_count", "opponent_bishop_count", "own_rook_count", "opponent_rook_count",
    "own_queen_count", "opponent_queen_count", "own_bishop_pair", "opponent_bishop_pair",
    "own_doubled_pawn_count", "opponent_doubled_pawn_count", "own_isolated_pawn_count",
    "opponent_isolated_pawn_count", "own_passed_pawn_count", "opponent_passed_pawn_count",
    "own_king_zone_attacked_count", "opponent_king_zone_attacked_count",
    "own_pawn_shield_count", "opponent_pawn_shield_count",
    "own_castling_rights_count", "opponent_castling_rights_count",
    "own_pinned_piece_count", "opponent_pinned_piece_count",
    "own_attacked_piece_count", "opponent_attacked_piece_count",
    "own_hanging_piece_count", "opponent_hanging_piece_count",
    "own_hanging_material_cp", "opponent_hanging_material_cp",
    "own_rooks_on_open_files_count", "opponent_rooks_on_open_files_count",
    #game features: ratings and starting time
    "mover_elo", "opponent_elo", "rating_difference", "initial_seconds",
    #clock features (clocks before the move)
    "time_left_seconds", "opponent_time_left_seconds", "time_left_fraction", "clock_difference",
    "time_left_log1p", "low_time_10s", "last_move_seconds",
    #move number and the engine evaluation before the move
    "move_number", "pre_eval_cp"]

metadata = ["game_id", "source_pgn", "time_control"]

#a typo in the list above would silently lose a feature, so we check the count
assert len(predictors) == 60

In [ ]:
#the columns we keep, in this order
keep = ["y"] + predictors + ["pre_eval_cp_missing"] + metadata

#show which columns we remove (empty if 01_cleaning already saved only these columns)
print("removed:", [col for col in master.columns if col not in keep])

#keep only the allowed columns; this also fails with an error if one of them is missing
master = master[keep]

#the columns that give away the target must be gone
assert not {"eval_loss_cp", "played_eval_cp", "played_mate"} & set(master.columns)

Every feature must be stored as a number (deck 06: "inconsistent data types"). Only the three metadata columns may be text.

In [ ]:
#columns that are not numbers; we expect exactly the three metadata columns
print(master.select_dtypes(exclude="number").columns.tolist())

## 3. Time control as dummies

The three time controls (1+0, 5+0, 10+0) are categories without a meaningful distance: a 10+0 game is not "ten times" a 1+0 game. Deck 06 says: categories without an order (nominal) get **one-hot encoding**, not numbers. This is why we do not use `initial_seconds` (60, 300, 600) as the time-control feature.

We use `pd.get_dummies` as in notebook 204. With `drop_first=True` one category is left out. Otherwise the dummies of all three categories add up to 1 in every row, exactly like the column of ones for the intercept, and the coefficients are not unique (deck 06, "one-hot encoding and the intercept"; the dummy trap). The left-out category is the reference: its effect is in the intercept.

There are only three fixed time controls, so this step learns nothing from the data and may happen before the split.

In [ ]:
#positions per time control
master["time_control"].value_counts()

In [ ]:
#one 0/1 column per time control, except the first one (drop_first=True)
#prefix="tc" names the columns tc_<time control>, dtype=int stores 0/1 instead of True/False
tc_dummies = pd.get_dummies(master["time_control"], prefix="tc", drop_first=True, dtype=int)

#add the dummy columns to master; we keep time_control itself for tables
master = pd.concat([master, tc_dummies], axis=1)

tc_cols = tc_dummies.columns.tolist()
print("dummy columns:", tc_cols)

`get_dummies` sorts the categories as text, so `"300+0"` comes before `"60+0"` and `"600+0"`. The output above shows which time control was left out: that one is the reference category.

## 4. Split by game into training and test games

All positions of one game belong together: same players, one position follows from the other. If we split the rows at random, neighbouring moves of the same game would end up in the training and in the test data, and the test error would be too optimistic (deck P1, "by group"; notebook 108b, warning before K-fold). So we keep each game whole.

We use `train_test_split` from notebook 108a, but on the list of **game ids** instead of the rows. Then every game lands on one side only.
- `test_size=0.2`: 20% of the games are test games (README: 80/20 split).
- `stratify=games["source_pgn"]`: each time control gets the same share of test games (README: "balancing time controls").
- `random_state=72`: a fixed seed, as in the course notebooks, so that everybody gets the same split.

**Team decision:** this is the simplest split that is close to the course. The earlier version in the models notebook used `StratifiedGroupKFold`, which also balances the blunder rate but is not in the course notebooks. With 9,000 games, the blunder rates of the two sides come out almost the same anyway (see the check below).

In [ ]:
#one row per game: its id and its file (= time control)
games = master[["game_id", "source_pgn"]].drop_duplicates()

#split the game ids, not the rows; stratify keeps the share of each time control equal
train_games, test_games = train_test_split(games["game_id"], test_size=0.2,
                                           stratify=games["source_pgn"], random_state=72)

#all positions of the training games, and all positions of the test games
train_df = master[master["game_id"].isin(train_games)]
test_df = master[master["game_id"].isin(test_games)]

In [ ]:
#sanity checks: no game on both sides, and no position lost
assert set(train_df["game_id"]).isdisjoint(test_df["game_id"])
assert len(train_df) + len(test_df) == len(master)

for name, d in [("train", train_df), ("test", test_df)]:
    print(f"{name}: {d['game_id'].nunique()} games, {len(d)} positions, blunder rate {d['y'].mean():.3f}")

In [ ]:
#games per time control on both sides
pd.DataFrame({"train games": train_df.groupby("source_pgn")["game_id"].nunique(),
              "test games": test_df.groupby("source_pgn")["game_id"].nunique()})

We save the two parts and the list of game ids with their side (README: "freeze an 80/20 split of whole games... save game IDs"). `03_models.ipynb` reads `train.csv` for the cross-validation and `test.csv` only once, at the very end.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

#the frozen split: one row per game with its side
split_ids = pd.concat([pd.DataFrame({"game_id": train_games, "split": "train"}),
                       pd.DataFrame({"game_id": test_games, "split": "test"})])
split_ids.to_csv("../data/processed/split_game_ids.csv", index=False)

print(f"saved {len(train_df)} training and {len(test_df)} test positions")

**From here on we only use `train_df`.** The test games stay untouched until the final test in `03_models.ipynb`. Every choice below (how to fill gaps, what to do with outliers) is made on the training games, so the test error stays honest (notebook 209).

## 5. Missing values

Deck 06 lists four options for missing values: remove columns, remove rows, impute, or recode with an indicator (notebook 204, appendix). First we look at where values are missing.

In [ ]:
#share of missing values per feature (training games only), largest first
missing = train_df[predictors].isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3)

Why values are missing (from `01_cleaning`):
- **Clocks** (`time_left_seconds`, `opponent_time_left_seconds`, `last_move_seconds` and the features made from them): a player's clock is unknown on their first turn, and the previous-move duration on their first two turns. `01_cleaning` removed all positions after a suspicious clock, so no other clock gaps should remain.
- **`pre_eval_cp`**: Stockfish sees a forced mate, so there is no centipawn value. `pre_eval_cp_missing` marks these positions.

Our choice:
- We do **not remove rows**: missing clocks are normal at the start of every game, and mate positions have a valid label. Deck 06 warns against dropping rows unless it is justified.
- We **impute with the median** of the training data and add a 0/1 indicator "was missing" (section 8). The indicator keeps the information that a value was missing, as `ever_delinquent` in notebook 204.

Why the median and not the mean? The mean reacts to extreme values, the median does not (notebook 106, task 11). The table compares both.

In [ ]:
#mean and median of the features with missing values (training games only)
cols_with_missing = missing[missing > 0].index
train_df[cols_with_missing].agg(["mean", "median"]).T.round(2)

Where mean and median are far apart, the distribution is skewed and the median is the more robust filling value.

Are positions with a missing evaluation different? The blunder rate by `pre_eval_cp_missing` shows it.

In [ ]:
#blunder rate and number of positions with (1) and without (0) an evaluation in centipawns
train_df.groupby("pre_eval_cp_missing")["y"].agg(["mean", "size"]).round(3)

## 6. Distributions and outliers

Deck 06 lists "lack of data exploration" as a mistake: look at the distributions before modeling. We plot histograms of features where we expect long tails.

In [ ]:
plot_cols = ["time_left_seconds", "time_left_log1p", "last_move_seconds",
             "pre_eval_cp", "own_hanging_material_cp", "legal_move_count"]

#a grid of 2 x 3 plots; axes.ravel() turns the grid into a simple list of six plots
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), plot_cols):
    #dropna(): a histogram cannot draw missing values
    ax.hist(train_df[col].dropna(), bins=50)
    ax.set_title(col)
fig.tight_layout()

`time_left_log1p` is ln(1 + seconds left). It was built in `Complexity.py` and is a log transform as in notebook 106, task 4 (the Box-Cox case with λ = 0). Compare it with `time_left_seconds`: the log stretches the crowded low values, where time trouble happens.

Next we count potential outliers with **Tukey's fences** (deck 06, notebook 106, task 9): a value is flagged if it lies more than 1.5 interquartile ranges below the first quartile or above the third quartile. The function is the one from the solution of task 9.

In [ ]:
#number of values below and above Tukey's fences (solution of notebook 106, task 9)
def tukey_counts(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int((s < q1 - k * iqr).sum()), int((s > q3 + k * iqr).sum())

#one row per feature with its two counts
tukey = pd.DataFrame([tukey_counts(train_df[col].dropna()) for col in predictors],
                     index=predictors, columns=["below", "above"])

#only the features with flagged values, most flagged first
tukey["share"] = ((tukey["below"] + tukey["above"]) / len(train_df)).round(3)
tukey[tukey["share"] > 0].sort_values("share", ascending=False)

**Flagged does not mean wrong** (solution of notebook 106, task 9). Our flagged values are real chess positions: a clearly won position (+1500 cp), many hanging pieces, a player with much more time. They are not data errors; `01_cleaning` already removed the impossible values (negative clocks, clocks going up).

Our choice: we **keep** all values.
- Trees (random forest, boosting) split on thresholds, so extreme values do not distort them (deck 06).
- For logistic regression we standardize (section 8). Very large `pre_eval_cp` values remain, so we look at them in section 7.

## 7. Blunder rate across the evaluation and the clock

How does the blunder rate change with a feature? As in the extra notebook 211 (section 4, "observed share of defaults in 20 groups"), we cut a feature into 20 groups of equal size and plot the share of blunders in each group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["pre_eval_cp", "time_left_log1p"]):
    #20 groups with about the same number of positions; duplicates="drop" merges equal group edges
    groups = pd.qcut(train_df[col], 20, duplicates="drop")
    #share of blunders and mean feature value in each group (positions with a missing value are left out)
    share = train_df["y"].groupby(groups, observed=True).mean()
    mid = train_df[col].groupby(groups, observed=True).mean()

    ax.scatter(mid, share, color="black", s=15)
    ax.axhline(train_df["y"].mean(), color="grey", linestyle="--", label="blunder rate of all positions")
    ax.set_xlabel(col)
    ax.set_ylabel("share of blunders")
    ax.legend()
fig.tight_layout()

What to look for:
- **Left plot.** If the blunder rate is high both in balanced positions and in clearly decided ones, the relation is not a straight line. A logistic regression has one coefficient for `pre_eval_cp` and can only draw one S-shaped curve, rising or falling (notebook 211). Trees can follow any shape.
- **Right plot.** Low `time_left_log1p` means little time left. A rising blunder rate on the left side of the plot shows time trouble.

Write down what the plots show; this is a good result for the presentation. Note for the report: `pre_eval_cp` is also part of the label formula (`eval_loss_cp = best_cp - played_cp`), so part of its predictive power comes from how the label is built.

We do **not** transform `pre_eval_cp` here. Two options from the course, if the team wants them later:
- clip it at ±1000 cp (winsorizing, deck 06), a fixed chess rule that learns nothing from the data;
- splines in the pipeline (extra notebook 211, section 4), which can draw a U-shape.

## 8. Feature groups

The models compare three feature groups (README):
- **M1**: ratings, time control, clocks, move number,
- **M2**: M1 plus move and board features,
- **M3**: M2 plus the engine evaluation before the move.

Some of the 60 predictors follow exactly from others, for example `rating_difference = mover_elo - opponent_elo`. Deck 06 and notebook 106 (task 8) show the problem: if one column is a sum or a difference of others, the design matrix with the column of ones loses rank, $X^\top X$ is not invertible, and the coefficients of a linear model are not unique. We check this with the function `design_rank` from notebook 106, task 8.

In [ ]:
#rank and number of columns of the design matrix [1, X] (notebook 106, task 8)
def design_rank(X):
    A = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    return np.linalg.matrix_rank(A), A.shape[1]

#the rank needs numbers without gaps, so we use the training positions without any missing value
complete = train_df.dropna()

#all 60 predictors plus the time-control dummies: is the rank smaller than the number of columns?
print("all predictors + dummies (rank, columns):", design_rank(complete[predictors + tc_cols]))

The rank is smaller than the number of columns: some columns follow exactly from others. These identities are the reason (as part 3 of task 8 in notebook 106). `np.isclose` compares with a tiny tolerance, because clock values are decimal numbers.

In [ ]:
c = complete

#ratings: the difference follows from the two ratings
print(np.isclose(c["rating_difference"], c["mover_elo"] - c["opponent_elo"]).all())

#clocks: the difference follows from the two clocks
print(np.isclose(c["clock_difference"], c["time_left_seconds"] - c["opponent_time_left_seconds"]).all())

#moves: every legal move is a capture, a check that is no capture, or a quiet move
print(np.isclose(c["legal_move_count"],
                 c["capture_move_count"] + c["checking_move_count"] - c["checking_capture_count"]
                 + c["quiet_move_count"]).all())

#material: pieces are worth 100 (pawn) to 900 (queen); own minus opponent material
print(np.isclose(c["material_balance_cp"],
                 c["own_non_pawn_material_cp"] + 100 * c["own_pawn_count"]
                 - c["opponent_non_pawn_material_cp"] - 100 * c["opponent_pawn_count"]).all())

#material: total material of both players
print(np.isclose(c["total_material_cp"],
                 c["own_non_pawn_material_cp"] + 100 * c["own_pawn_count"]
                 + c["opponent_non_pawn_material_cp"] + 100 * c["opponent_pawn_count"]).all())

`initial_seconds` takes only three values, one per time control, so it is a linear combination of the column of ones and the two time-control dummies as well.

We therefore leave out one column of each identity and use the time-control dummies instead of `initial_seconds`. These are the feature groups of the models notebook, with the dummies added to block A:
- left out: `opponent_elo`, `initial_seconds`, `time_left_seconds`, `opponent_time_left_seconds`, `quiet_move_count` and the piece counts (`own_pawn_count`, ..., `opponent_queen_count`; the material columns and queen counts carry this information).

In [ ]:
#block A: ratings, time control, clocks, move number
block_A = ["mover_elo", "rating_difference", "time_left_log1p", "time_left_fraction",
           "low_time_10s", "clock_difference", "last_move_seconds", "move_number"] + tc_cols

#block B: move and board features
block_B = ["legal_move_count", "capture_move_count", "checking_move_count",
           "checking_capture_count", "promotion_move_count", "castling_move_count",
           "in_check", "checker_count",
           "material_balance_cp", "total_material_cp",
           "own_non_pawn_material_cp", "opponent_non_pawn_material_cp",
           "own_queen_count", "opponent_queen_count",
           "own_bishop_pair", "opponent_bishop_pair",
           "own_doubled_pawn_count", "opponent_doubled_pawn_count",
           "own_isolated_pawn_count", "opponent_isolated_pawn_count",
           "own_passed_pawn_count", "opponent_passed_pawn_count",
           "own_king_zone_attacked_count", "opponent_king_zone_attacked_count",
           "own_pawn_shield_count", "opponent_pawn_shield_count",
           "own_castling_rights_count", "opponent_castling_rights_count",
           "own_pinned_piece_count", "opponent_pinned_piece_count",
           "own_attacked_piece_count", "opponent_attacked_piece_count",
           "own_hanging_piece_count", "opponent_hanging_piece_count",
           "own_hanging_material_cp", "opponent_hanging_material_cp",
           "own_rooks_on_open_files_count", "opponent_rooks_on_open_files_count"]

#block C: the engine evaluation before the move (its missing indicator is added in section 8b)
block_C = ["pre_eval_cp"]

M1 = block_A
M2 = block_A + block_B
M3 = block_A + block_B + block_C

#full rank: the number of columns equals the rank for every group
for name, cols in [("M1", M1), ("M2", M2), ("M3", M3)]:
    print(name, "(rank, columns):", design_rank(complete[cols]))

If the rank equals the number of columns for M1, M2 and M3, every coefficient is unique. If a rank is smaller, another column follows from the others in our data; then look for it as above and leave it out.

*Copy these three blocks into `03_models.ipynb`; the only change to the earlier version is `+ tc_cols` in block A.*

## 8b. The steps that learn from the data

Two steps learn numbers from the data:
1. **Imputation**: fill each missing value with the **median** of the training data (notebook 106, task 11).
2. **Standardization**: $x_{\text{std}} = (x - \hat\mu) / \hat\sigma$ with the mean $\hat\mu$ and the standard deviation $\hat\sigma$ of the training data (deck 06, notebook 108b).

"Estimate, then freeze" (deck 06): the median, $\hat\mu$ and $\hat\sigma$ are estimated once on the training data and then applied unchanged to new data. In the cross-validation, the "training data" are the training folds, so these numbers must be estimated again in every fold. Otherwise information from the validation fold leaks into training (deck P1, "preprocessing fitted on all data").

**By hand first.** As in notebook 108b, we compute the result by hand and with scikit-learn, and check that both give the same numbers. Here we use the training games as an example.

In [ ]:
#three example columns with missing values
X_ex = train_df[["time_left_log1p", "last_move_seconds", "pre_eval_cp"]]

#1) by hand: fill gaps with the median (fill_missings in notebook 106), then (x - mean) / sd (notebook 108b)
filled = X_ex.fillna(X_ex.median())
by_hand = (filled - filled.mean()) / filled.std(ddof=0)   #ddof=0 divides by n, as scikit-learn does

#2) with scikit-learn: SimpleImputer fills with the median, StandardScaler standardizes
#set_output(transform="pandas") keeps a DataFrame with column names (as in notebook 108b)
imputer = SimpleImputer(strategy="median").set_output(transform="pandas").fit(X_ex)
X_filled = imputer.transform(X_ex)
scaler = StandardScaler().set_output(transform="pandas").fit(X_filled)

print("Same result:", np.allclose(scaler.transform(X_filled), by_hand))

`SimpleImputer` is not used in the course notebooks, but the check shows that it does exactly what `fill_missings(df_col, median=True)` in notebook 106 does.

**The same steps as a pipeline.** In the models notebook, the two steps are put into a pipeline (notebook 209, extra notebook 211). `cross_val_score` and `cross_validate` then fit the pipeline on the training folds only and apply it to the validation fold, automatically in every fold.

- `add_indicator=True` adds a 0/1 column "was missing" for every column that has missing values. For `pre_eval_cp` this is the same as `pre_eval_cp_missing`, so we do not put `pre_eval_cp_missing` into the feature lists as well (it would be the same column twice).
- Logistic regression and neural network: use `make_prep()` in front of the model, for example `make_pipeline(make_prep(), LogisticRegression())`.
- Random forest and boosting: no `make_prep()`. Trees split on thresholds and need no scaling (deck 06, notebook 210); scikit-learn's trees handle missing values themselves.

In [ ]:
#the preprocessing that is learned from the data; used in 03_models.ipynb in front of the model
def make_prep():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())

#example: fitted on the training games, the indicator adds one 0/1 column per feature with gaps
prep = make_prep().fit(train_df[M3])
print(f"{len(M3)} features in, {prep.transform(train_df[M3]).shape[1]} columns out")

## 9. Preprocessing summary

| Step | Where | Learned from data? | Course |
|---|---|---|---|
| Remove unusable rows, mask suspicious clocks, drop rows without `y` | `01_cleaning` | no | deck 06 (inconsistent data) |
| Indicator `pre_eval_cp_missing` | `01_cleaning` | no | notebook 204, appendix (recode) |
| Log transform `time_left_log1p`, time share `time_left_fraction` | `Complexity.py` | no | notebook 106, task 4 |
| Keep only `y`, predictors and three metadata columns | section 2 | no | deck P1 (leakage) |
| Time control as dummies, `drop_first=True` | section 3 | no (three fixed categories) | deck 06, notebooks 106 and 204 |
| Split by game, 80/20, balanced by time control | section 4 | no | deck P1, notebooks 108a and 209 |
| Keep outliers | section 6 | no | notebook 106, task 9 |
| Leave out columns that follow from others | section 8 | no | deck 06, notebook 106, task 8 |
| Median imputation + missing indicator | `make_prep()`, fitted in each fold | yes | notebook 106, task 11; notebook 204 |
| Standardization | `make_prep()`, fitted in each fold | yes | deck 06, notebook 108b |

Files for `03_models.ipynb`: `../data/processed/train.csv`, `../data/processed/test.csv` and `../data/processed/split_game_ids.csv`.